# T60 AUV 训练配置

这个 Notebook 是训练实验的唯一人工配置入口。这里只设置流速、环境效应、Domain Randomization 强度和训练参数；配置校验、文件生成、进程管理与 Isaac worker 全部由 Python 模块实现。

In [ ]:
from pathlib import Path
import json
import sys

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / 'simulation/isaac').is_dir():
    raise RuntimeError('请从 isaac-auv-env 仓库根目录启动 train.ipynb。')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from simulation.isaac.training import (
    build_default_campaign, campaign_payload, launch_campaign,
    materialize_training_profiles, render_campaign_status, stop_campaign,
)
from simulation.isaac.trajectory.experiment import build_train_command, display_command


## 唯一数值配置区

确定性水流和水池效应写入本次运行的环境快照；所有 DR 数值在 `DR` 中显式列出。生成文件位于 Git 忽略的 `simulation/isaac/rlpolicy/_configs/<RUN_NAME>/`。

In [ ]:
ISAACLAB_ROOT = Path.home() / 'IsaacLab'
RUN_NAME = 't60_policy_6'
MLP_ARCHITECTURE = 'mlp_history_5'
REWARD_PROFILE = 'policy_6'
SEED = 42
NUM_ENVS = 1024
MAX_ITERATIONS = 500
ROLLOUT_STEPS = 256
SEGMENT_ITERATIONS = 25

# 确定性水流：世界系常值、周期流和可选空间流场。
HYDRODYNAMICS = {
    'water_current_w': [0.0, 0.0, 0.0],
    'water_current_periodic_enabled': True,
    'water_current_periodic_amplitude_w': [0.04, 0.025, 0.008],
    'water_current_periodic_period_s': [18.0, 27.0, 11.0],
    'water_current_periodic_phase_rad': [0.0, 1.5707963267948966, 3.141592653589793],
    'water_current_field_enabled': False,
    'water_current_field_bounds': [-7.0, 7.0, -7.0, 7.0, -15.0, -1.0],
    'water_current_field_shape': [1, 1, 1],
    'water_current_field_values': [[0.0, 0.0, 0.0]],
}

POOL_BOUNDARY = {
    'enabled': True, 'bounds': [-7.0, 7.0, -7.0, 7.0, -15.0, -1.0],
    'effect_distance': 0.9, 'damping_scale_at_boundary': 1.35,
    'added_mass_scale_at_boundary': 1.12, 'thrust_scale_at_boundary': 0.90,
}
FREE_SURFACE = {
    'enabled': True, 'surface_z': -1.0, 'effect_distance': 0.65,
    'heave_damping_scale': 1.30, 'roll_pitch_damping_scale': 1.15,
    'added_mass_scale': 1.10, 'buoyancy_scale': 0.96, 'thrust_scale': 0.92,
    'sloshing_enabled': True, 'sloshing_pool_bounds': [-7.0, 7.0, -7.0, 7.0],
    'sloshing_water_depth': 14.0, 'sloshing_mode_numbers': [[1, 0], [0, 1]],
    'sloshing_amplitudes_m': [0.018, 0.012],
    'sloshing_phases_rad': [0.0, 1.5707963267948966],
    'sloshing_depth_axis_sign': -1.0,
}

# 所有训练 DR 数值。零宽范围表示该项不随机化。
DR = {
    'use_custom_randomization': True,
    'enabled_features': ['rigid_body', 'current', 'hydrodynamics', 'actuators', 'battery'],
    'com_to_cob_offset_radius': 0.0,
    'volume_range': [0.011304505834, 0.011304505834],
    'mass_range': [11.301, 11.301],
    'payload_samples': [],
    'thruster_command_delay_steps_range': [0, 0],
    'thruster_max_command_rate_range': [0.0, 0.0],
    'thruster_command_resolution_range': [0.0, 0.0],
    'thruster_command_dropout_probability_range': [0.0, 0.0],
    'thruster_wake_loss_coefficient_scale_range': [1.0, 1.0],
    'thruster_reaction_torque_coeff_scale_range': [1.0, 1.0],
    'damping_speed_linear_scale_range': [1.0, 1.0],
    'damping_speed_quadratic_scale_range': [1.0, 1.0],
    'battery_voltage_range': [16.0, 16.0],
    'battery_voltage_drop_per_s_range': [0.0, 0.0],
    'disturbance_curriculum': True,
    'disturbance_curriculum_stage_steps': [9750, 22500, 40500, 57000],
    'water_current_smooth': True,
    'water_current_tau_range': [8.0, 24.0],
    'water_current_max_by_stage': [0.0, 0.05, 0.10, 0.15, 0.20],
    'water_current_vertical_max_by_stage': [0.0, 0.01, 0.02, 0.025, 0.03],
    'water_current_variation_std_by_stage': [0.0, 0.004, 0.008, 0.012, 0.016],
    'damping_scale_by_stage': [0.0, 0.0, 0.15, 0.25, 0.30],
    'added_mass_log_std_by_stage': [0.0, 0.0, 0.05, 0.08, 0.12],
    'thruster_scale_by_stage': [0.0, 0.0, 0.0, 0.10, 0.15],
    'thruster_tau_scale_by_stage': [0.0, 0.0, 0.0, 0.25, 0.50],
    'additional_hydrodynamics_scale_by_stage': [0.0, 0.0, 0.35, 0.70, 1.0],
}


In [ ]:
profiles = materialize_training_profiles(
    RUN_NAME, hydrodynamics=HYDRODYNAMICS, pool_boundary=POOL_BOUNDARY,
    free_surface=FREE_SURFACE, randomization=DR,
)
campaign = build_default_campaign(
    isaaclab_root=ISAACLAB_ROOT, mlp_architecture=MLP_ARCHITECTURE,
    reward_profile=REWARD_PROFILE, seed=SEED, num_envs=NUM_ENVS,
    run_name=RUN_NAME, max_iterations=MAX_ITERATIONS,
    rollout_steps_per_env=ROLLOUT_STEPS, segment_iterations=SEGMENT_ITERATIONS,
    environment_profile=profiles.environment,
    domain_randomization_spec=profiles.randomization,
)
print(display_command(build_train_command(campaign.experiment, campaign.train), cwd=campaign.experiment.isaaclab_root))
print(json.dumps(campaign_payload(campaign), indent=2))


## 执行动作

默认只预览。改为 `start`、`status` 或 `stop` 后重新运行本单元。

In [ ]:
ACTION = 'preview'  # preview | start | status | stop
RESTART_GATE = False

if ACTION == 'preview':
    print('仅预览；未启动训练。')
elif ACTION == 'start':
    pid, log_path = launch_campaign(campaign, restart=RESTART_GATE)
    print(f'PID={pid}\nlog={log_path}')
elif ACTION == 'status':
    print(render_campaign_status(campaign))
elif ACTION == 'stop':
    stopped, removed = stop_campaign(campaign, clean_stale=True)
    print('stopped:', stopped, 'removed:', [path.name for path in removed])
else:
    raise ValueError(f'未知 ACTION: {ACTION}')
